# Rotation evolution

実装は `rot_evol.py` にまとめ、このノートブックでは設定と実行のみを行う。結果は `result/<LABEL>/` に保存される。

In [1]:
import os
os.environ["JAX_NUM_CPU_DEVICES"] = "4"
import importlib
from pathlib import Path
import arviz as az
import jax
import numpyro
from numpyro.infer import MCMC, NUTS
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
numpyro.set_host_device_count(4)
print(jax.devices())
print(jax.device_count())
import rot_evol
importlib.reload(rot_evol)
from rot_evol import (
    IsochronePosterior,
    RotEvol,
    frame_ids_from_observation_log,
    save_figures,
    save_inference_data,
    save_logage_histogram,
    selection_label,
)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3)]
4


In [2]:
ISO_NSAMPLES = 10000
NUM_WARMUP = 4000
NUM_SAMPLES = 4000
NUM_CHAINS = 4
TARGET_ACCEPT_PROB = 0.8 
MAX_TREE_DEPTH = 12
RNG_SEED = 0
PREDICT_P_ROT = False
OBSERVATION_LOG = Path("observation_log.csv")
LOG_QUERY = ""
# MASS_RANGE = (0.9, 1.1)
MASS_RANGE = (1.1, 1.3)
FEH_RANGE = (None, -0.1)
# FEH_RANGE = (-0.1, None)
BASE_LABEL = selection_label(MASS_RANGE, FEH_RANGE)
LABEL = f"{BASE_LABEL}_lognormage_predict_age"
print(f"Result label: {LABEL}")

frame_ids = frame_ids_from_observation_log(
    OBSERVATION_LOG,
    query=LOG_QUERY,
    exclude_tags=("Trash",),
    keep_highest_sn_per_object=True,
    sn_column="Count (e-)",  
)
re = RotEvol.from_analysis_results(frame_ids, refresh_cache = False).select(
    mass_range=MASS_RANGE, feh_range=FEH_RANGE
)
isochrone_posterior = (
    IsochronePosterior.from_analysis_results(
        re.frame_id, nsamples=ISO_NSAMPLES, refresh_cache=False,
    )
)
print(f"{re.N} stars: {re.frame_id.tolist()}")

Result label: mass1.1-1.3_feh-0.1low_lognormage_predict_age


Loading analysis results:   0%|          | 0/225 [00:00<?, ?frame/s]

KeyboardInterrupt: 

In [ ]:
# kernel = NUTS(re.numpyro_model, target_accept_prob=TARGET_ACCEPT_PROB)
# model_args = (isochrone_posterior,)
# dense_mass=[("logage_prior_mu", "loglogage_prior_sigma","a","logb")]

model_args = (isochrone_posterior, True, PREDICT_P_ROT)

dense_mass=[("a","logb","lognorm_age_mu","lognorm_age_sigma")]
kernel = NUTS(
    re.power_model,
    target_accept_prob=TARGET_ACCEPT_PROB,
    dense_mass=dense_mass,
    max_tree_depth=MAX_TREE_DEPTH,
    )
mcmc = MCMC(
    kernel,
    num_warmup=NUM_WARMUP,
    num_samples=NUM_SAMPLES,
    num_chains=NUM_CHAINS,
    # chain_method="parallel",
    chain_method="sequential",
)
mcmc.run(jax.random.PRNGKey(RNG_SEED), *model_args)

idata = az.from_numpyro(mcmc)
save_inference_data(re, idata, LABEL)
save_figures(
    re,
    idata,
    LABEL,
    isochrone_posterior=isochrone_posterior,
    predict_P_rot=PREDICT_P_ROT,
)
logage_hist_path = save_logage_histogram(
    re, Path("result") / LABEL / "logage_hist.png", idata=idata,
)

warmup:   0%|          | 6/8000 [00:20<7:27:18,  3.36s/it, 7 steps of size 2.19e-03. acc. prob=0.34]


KeyboardInterrupt: 

In [ ]:
isochrone_posterior.Nsamples

10000

In [ ]:
re.vsini

array([ 25.6176805 ,  25.36226837,   1.48093464,  15.40829347,
        16.71000307,   4.78760365,  12.47303886,   9.68405282,
         2.84336355,   7.86097849,  22.88886267,  31.97118329,
         6.44086063,  28.20852713,   8.24081488,   4.89096745,
        10.2910855 ,   8.23896927,   7.87723816,   9.20048946,
         6.2617719 ,  28.17908805,   5.27646655,  16.96023754,
         1.36625404,   8.77663519,   1.90327157,   7.33426423,
       106.02266242,   7.66209901,  61.1212799 ,   8.11945642,
         3.35860151])